# 1. Raw data 가져오기

In [1]:
from datasets import load_dataset
dataset = load_dataset("Cartinoe5930/raw_text_synthetic_dataset_50k", split = "train")

# 2. 역색인(using BM25)

In [2]:
from kiwipiepy import Kiwi
from collections import defaultdict
import math

In [ ]:
class InvertedIndex:
    def __init__(self):
        self.reset()

    def reset(self):
        self.index = defaultdict(dict)
        self.kiwi = Kiwi()
        self.document_lengths = {}
        self.total_documents = 0
        self.average_document_length = 0
        self.documents = {}

    def tokenize(self, text):
        return [token.form for token in self.kiwi.tokenize(text)]

    def add_document(self, doc_id, question, answer):
        tokens = self.tokenize(question)
        self.document_lengths[doc_id] = len(tokens)
        self.total_documents += 1
        self.documents[doc_id] = {'question': question, 'answer': answer}

        for token in set(tokens):
            if doc_id not in self.index[token]:
                self.index[token][doc_id] = 0
            self.index[token][doc_id] += tokens.count(token)

        self.average_document_length = sum(self.document_lengths.values()) / self.total_documents

    def calculate_bm25_score(self, query_tokens, doc_id):
        k1 = 1.5
        b = 0.75
        score = 0

        for token in query_tokens:
            if token not in self.index or doc_id not in self.index[token]:
                continue

            tf = self.index[token][doc_id]
            df = len(self.index[token])
            idf = math.log((self.total_documents - df + 0.5) / (df + 0.5) + 1)

            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * self.document_lengths[doc_id] / self.average_document_length)
            score += idf * numerator / denominator

        return score

    def search(self, query, k=5):
        query_tokens = self.tokenize(query)
        scores = defaultdict(float)

        for token in query_tokens:
            if token in self.index:
                for doc_id in self.index[token]:
                    scores[doc_id] += self.calculate_bm25_score(query_tokens, doc_id)

        top_k = sorted(scores.items(), key=lambda x: x[1], reverse = True)[:k]
        return [(doc_id, score, self.documents[doc_id]) for doc_id, score in top_k]

In [ ]:
# add docs
from tqdm import tqdm

index = InvertedIndex()
for idx, data in enumerate(tqdm(dataset)):
    question = data['question']
    answer = data['response']
    index.add_document(idx, question, answer)

# 3. 합성 데이터 생성

In [ ]:
import openai
from dotenv import load_dotenv
import os

dotenv_path = os.path.join(os.getcwd(), '.env')
load_dotenv(dotenv_path)

openai_key = os.getenv('OPENAI_TEAM_API_KEY')
client = openai.OpenAI(api_key=openai_key)

In [ ]:
system_prompt = """
You are given two pairs of reference questions and reference answers.
Your role is a questioner who make a new question.
When making your questions, consider the following.
1. New Question must require choices such as 'Which is right', 'Which is not right', 'Which is most appropriate', and 'Which is not most appropriate'.
2. You have to make 5 choices, 1 answer choice and 4 wrong choices.
3. The choices must be generated in association with one of several keywords in the reference question and answer.
4. The wrong answer and the right answer are confused, but the wrong answer must be a clear wrong answer.
5. The choices does not deviate from the subject of the problem, but it must be different.
6. The choices sentence must be similar in length.
7. If a person is a financial expert, the person can solve the problem, but if the person is a beginner in financial knowledge, please make the problem with a difficulty that the person cannot solve because it is difficult.
8. Please don't create a problem that can be solved by reading other than the problem.
please write in Korean and you must write the answer on the last line.
"""

user_prompt ="""
### Reference
### Question 1: {}
### Answer 1: {}

### Question 2: {}
### Answer 2: {}

### New Question : 

"""

In [ ]:
import random
from tqdm import tqdm

new_questions = []
for idx, data in enumerate(tqdm(dataset, total=len(dataset))):
    question = data['question']
    answer = data['response']
    system_msg = {'role':'system', 'content': system_prompt}

    similar_questions = index.search(question, k=2)

    user_msg = {'role':'user', 'content': user_prompt.format(
        similar_questions[0][2]['question'],
        similar_questions[0][2]['answer'],
        similar_questions[1][2]['question'],
        similar_questions[1][2]['answer'])}

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages = [system_msg, user_msg],
    )

    result = response.choices[0].message.content
    new_questions.append(result)
    
    if idx % 100 == 0:
        print(result)
        print("*"*100)


In [ ]:
import pandas as pd

mcqa_data = []

for raw_question in new_questions:
    lines = [line.strip() for line in raw_question.split('\n') if line.strip() != '']
    question = lines[:-1]
    answer = lines[-1]
    
    for i in range(1, 6):
        if str(i) in answer:
            answer = i
            break

    item = {"question": question, 'answer': answer}
    mcqa_data.append(item)

df = pd.DataFrame(mcqa_data)

# 4. 합성 데이터 Quality Control

## 4-1 중복 정답 체크

In [ ]:
system_prompt ="""
You are a financial expert.
Read the following questions and choose the answer.
It could be multiple correct answers.
Just write a number or numbers.
"""

user_prompt="""
Question : {}
"""

In [ ]:
from tqdm import tqdm

gpt_answer = []
for _, row_item in tqdm(df.iterrows(), total=len(df)):
    answer = int(row_item['answer'])
    question = str(row_item['question'])

    system_msg = {'role': 'system', 'content':system_prompt}
    user_msg = {'role':'user', 'content':user_prompt.format(question)}

    response = client.chat.completions.create(
        model = 'gpt-4o-mini',
        messages = [system_msg, user_msg]
    )

    response_text = response.choices[0].message.content

    gpt_4o_answer = []
    for i in range(6):
        if str(i) in response_text:
            gpt_4o_answer.append(i)

    gpt_answer.append(gpt_4o_answer)

## 4-2 중복 정답 선택지 바꾸기

In [ ]:
fixed_system_prompt = """
Your role is to correct the options for financial matters. You will be given the problem and the option number that you will need to rewrite. Please correct the given option number's sentence for your question by meeting the following conditions.
1. The selection must not deviate from the subject of the problem.
2. It should undoubtedly be corrected with a perfect incorrect answer.
3. It must not be modified in the same sense as other fingerprints.
4. The length selected must be similar to the length of other options.
You must only write correct options which satisfy above conditions.
"""

fixed_user_prompt="""
Question: {}
fixed_numbers : {}
"""

In [ ]:
fixed_results = []
df['gpt_answer'] = gpt_answer

for idx, row_item in tqdm(df.iterrows(), total=len(df)):
    question = row_item['question']
    answer = row_item['answer']
    gpt_answer = row_item['gpt_answer']
    fixed_question = ''

    need_to_fix_numbers = [num for num in gpt_answer if num != answer]
    if len(need_to_fix_numbers) >= 1:
        fixed_numbers_str = ', '.join(map(str, need_to_fix_numbers))
        fixed_system_msg = {'role':'system', 'content': fixed_system_prompt}
        fixed_user_msg={'role':'user', 'content': fixed_user_prompt.format(question, fixed_numbers_str)}

        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages = [fixed_system_msg, fixed_user_msg]
        )

        result = response.choices[0].message.content
        question_lines = question.split('\n')
        for fixed_choice in result.split('\n'):
            fixed_choice_text = fixed_choice.strip()
                
            for i in range(1, 6):
                if fixed_choice_text.startswith(str(i)):
                    question_lines[i] = fixed_choice_text

        fixed_question = '\n'.join(question_lines)

    if fixed_question:
        new_item = {'question':fixed_question, 'answer':answer}
    else:
        new_item = {'question':question, 'answer':answer}

    fixed_results.append(new_item)

fixed_df = pd.DataFrame(fixed_results)

# 5. 데이터 전처리 & 저장

In [ ]:
import re

def preprocess_questions(question):
    question_lines = question.split('\n')

    number_to_eng = {
        '1. ': 'A. ',
        '2. ': 'B. ',
        '3. ': 'C. ',
        '4. ': 'D. ',
        '5. ': 'E. ',
    }

    new_question_lines = []
    for idx, line in enumerate(question_lines):
        new_line = ''
        for i in range(6):
            choice_string = f"{str(i)}. "
            if line.startswith(choice_string):
                new_line = re.sub(choice_string, number_to_eng[choice_string], line)
                break
        if new_line:
            new_question_lines.append(new_line)
        else:
            new_question_lines.append(line)
    
    question_text = "### 질문: " + new_question_lines[0]
    choice_text = "### 선택지:\n" + '\n'.join(new_question_lines[1:])
    result = question_text + "\n" + choice_text

    return result

def preprocess_answer(answer):
    number_to_eng = {
        1: 'A',
        2: 'B',
        3: 'C',
        4: 'D',
        5: 'E',
    }

    return number_to_eng[answer] 

In [ ]:
fixed_df['question'] = fixed_df['question'].apply(preprocess_questions)
fixed_df['answer'] = fixed_df['answer'].apply(preprocess_answer)
fixed_df.to_csv('datasets/fixed_sample_questions.csv', index=False)